# ValidEval V7 — S3 scientific BBH

Frozen T4×2 execution notebook. `fixture` validates the exact production config and exercises
the packaged mock path, but is always `NON_EVIDENCE_FIXTURE`. Real runs require two visible T4s,
the final V7 source tag, exact dataset/model revisions, and unchanged file hashes.

In [ ]:
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path(os.environ.get("VALIDEVAL_REPOSITORY_ROOT", ".")).resolve()
MODE = os.environ.get("VALIDEVAL_EXECUTION_MODE", "fixture").strip()
OUTPUT_ROOT = Path(os.environ.get("VALIDEVAL_NOTEBOOK_OUTPUT_ROOT", "kaggle_v7_outputs"))
CONFIGS = ['configs/runs_v7/bbh_s3_v7.yaml']
S4_ROUTE = os.environ.get("VALIDEVAL_S4_ROUTE", "maximum").strip().lower()
if S4_ROUTE == "fallback":
    CONFIGS = [config.replace("_s4_v7.yaml", "_s4_fallback_v7.yaml") for config in CONFIGS]
elif S4_ROUTE != "maximum":
    raise ValueError("VALIDEVAL_S4_ROUTE must be maximum or fallback")
REQUIREMENTS = ROOT / "requirements-kaggle-t4x2-v7.txt"
EXPECTED_REQUIREMENTS_SHA256 = 'e1f8c556b3c51054c51e2d329ea43e09befe4bbdf8e94f1bad873788219bb411'
assert hashlib.sha256(REQUIREMENTS.read_bytes()).hexdigest() == EXPECTED_REQUIREMENTS_SHA256

if MODE != "fixture" and importlib.util.find_spec("valideval") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(ROOT)], check=True)

from valideval.execution.notebook_v7 import run_notebook_config_v7
print(json.dumps({"mode": MODE, "configs": CONFIGS, "output_root": str(OUTPUT_ROOT)}, indent=2))

In [ ]:
RESULTS = [
    run_notebook_config_v7(ROOT / config, mode=MODE, output_root=OUTPUT_ROOT)
    for config in CONFIGS
]
for result in RESULTS:
    if MODE == "fixture":
        assert result["evidence_class"] == "NON_EVIDENCE_FIXTURE"
        assert result["production_config_validated"] is True
print(json.dumps(RESULTS, indent=2, sort_keys=True))

In [ ]:
print("Download every deterministic ZIP from:", OUTPUT_ROOT / "packages")
print("Local import command:")
print("python -m valideval ingest-and-analyze --input <downloaded-zip-or-directory>")
print("Resume by setting VALIDEVAL_EXECUTION_MODE=resume and rerunning this notebook.")